# Book-level segmentation: research

We currently bucket characters into **3 publication-series groups** (Fire & Blood, Dunk & Egg, ASOIAF) using date ranges from infobox `Born` / `Died` (see `03_execute.ipynb`). The date-based rule has two weaknesses:

1. Only ~50% of characters have a parseable date — the rest fall into `Unknown` and we use ASOIAF-book mentions in bio prose as a fallback.
2. The bucket is coarse: a Robert's Rebellion character (Brandon Stark, Aerys II) and a current-timeline character (Tyrion) both land in the same `A Song of Ice and Fire` bucket, even though the former never appears in any mainline novel.

AWOIAF pages encode book appearances **directly**: characters in *A Game of Thrones* get an h3 subsection `A Game of Thrones` under the `Recent Events` h2 (or under `History`, for older characters whose POV is via flashbacks). The `scrape_section_headings.ipynb` survey confirmed all the mainline ASOIAF book titles appear as h3 subsection headings across hundreds of character pages.

**This notebook tests whether we can lift those h3 subsections per-character to assign each character a *book set* — the explicit list of books they appear in.**

If coverage is good, this becomes a higher-fidelity replacement for the date-based bucket, and gives us per-book sub-segmentation inside ASOIAF (AGOT-only vs ADWD-only).

## 1. Setup + canonical book list

We enumerate every published Westeros book that could plausibly appear as a section heading. Per-book series mapping follows the publication grouping used in `03_execute.ipynb`.

In [28]:
from bs4 import BeautifulSoup
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import quote
import pandas as pd
import requests

BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
})

# Canonical books we'll look for as h2/h3 headings. Series ordering matches publication groups in 03_execute.ipynb.
BOOK_CANON = {
    # Fire & Blood era (book series, not pre-Conquest)
    'Fire & Blood':                  ('Fire & Blood', 0),
    'The Princess and the Queen':    ('Fire & Blood', 1),
    'The Rogue Prince':              ('Fire & Blood', 2),
    'The Sons of the Dragon':        ('Fire & Blood', 3),
    # Tales of Dunk & Egg novellas
    'The Hedge Knight':              ('Dunk & Egg', 4),
    'The Sworn Sword':               ('Dunk & Egg', 5),
    'The Mystery Knight':            ('Dunk & Egg', 6),
    'The She-Wolves of Winterfell':  ('Dunk & Egg', 7),
    # ASOIAF main series
    'A Game of Thrones':             ('ASOIAF', 8),
    'A Clash of Kings':              ('ASOIAF', 9),
    'A Storm of Swords':             ('ASOIAF', 10),
    'A Feast for Crows':             ('ASOIAF', 11),
    'A Dance with Dragons':          ('ASOIAF', 12),
    'The Winds of Winter':           ('ASOIAF', 13),
}
BOOK_LOOKUP = {k.lower(): k for k in BOOK_CANON}
print(f'Tracking {len(BOOK_CANON)} canonical book titles.')

Tracking 14 canonical book titles.


## 2. Per-character heading scrape

For each character page, walk the `mw-parser-output` div, collect every `<h2>` and `<h3>` headline, and keep the subset that matches a canonical book title (case-insensitive). We also record the parent h2 for each h3 — book headings under `Recent Events` vs `History` may behave differently.

In [29]:
def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def extract_book_signals(slug):
    """Returns {'ID': slug, 'books': sorted list of canonical book names,
    'h2_books': list of books appearing at h2 level,
    'recent_events_books': books listed under the Recent Events h2,
    'history_books': books listed under the History h2}."""
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException:
        return {'ID': slug, 'books': [], 'h2_books': [], 'recent_events_books': [], 'history_books': []}
    if resp.status_code != 200:
        return {'ID': slug, 'books': [], 'h2_books': [], 'recent_events_books': [], 'history_books': []}

    soup = BeautifulSoup(resp.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return {'ID': slug, 'books': [], 'h2_books': [], 'recent_events_books': [], 'history_books': []}

    all_books, h2_books, re_books, hist_books = set(), set(), set(), set()
    current_h2 = None
    for h in content.find_all(['h2', 'h3']):
        span = h.find('span', class_='mw-headline')
        if not span:
            continue
        title = span.get_text(strip=True)
        title_lower = title.lower()
        if h.name == 'h2':
            current_h2 = title_lower
            if title_lower in BOOK_LOOKUP:
                canonical = BOOK_LOOKUP[title_lower]
                all_books.add(canonical)
                h2_books.add(canonical)
        elif h.name == 'h3' and title_lower in BOOK_LOOKUP:
            canonical = BOOK_LOOKUP[title_lower]
            all_books.add(canonical)
            if current_h2 == 'recent events':
                re_books.add(canonical)
            elif current_h2 == 'history':
                hist_books.add(canonical)
    return {
        'ID': slug,
        'books': sorted(all_books, key=lambda b: BOOK_CANON[b][1]),
        'h2_books': sorted(h2_books, key=lambda b: BOOK_CANON[b][1]),
        'recent_events_books': sorted(re_books, key=lambda b: BOOK_CANON[b][1]),
        'history_books': sorted(hist_books, key=lambda b: BOOK_CANON[b][1]),
    }


## 3. Sample probe: 30 well-known characters spanning all 3 series

Hand-picked so we can sanity-check the output against what we *know* should be there (e.g. Eddard Stark should have AGOT, ACOK, ASOS; Aegon I should have *Fire & Blood*; Duncan should have D&E novellas).

In [30]:
PROBE_SLUGS = [
    # ASOIAF main series — POV characters
    'Eddard_Stark', 'Catelyn_Stark', 'Tyrion_Lannister', 'Jon_Snow', 'Daenerys_Targaryen',
    'Sansa_Stark', 'Arya_Stark', 'Bran_Stark', 'Cersei_Lannister', 'Jaime_Lannister',
    'Samwell_Tarly', 'Theon_Greyjoy', 'Davos_Seaworth', 'Brienne_of_Tarth', 'Areo_Hotah',
    # ASOIAF non-POV but central
    'Joffrey_Baratheon', 'Robert_I_Baratheon', 'Stannis_Baratheon', 'Petyr_Baelish',
    # Fire & Blood era
    'Aegon_I_Targaryen', 'Maegor_I_Targaryen', 'Jaehaerys_I_Targaryen', 'Aegon_II_Targaryen',
    # Dunk & Egg
    'Duncan_the_Tall', 'Aegon_V_Targaryen', 'Raymun_Fossoway',
    # Ambiguous (Robert's Rebellion era: pre-novel but heavily flashbacked)
    'Rhaegar_Targaryen', 'Aerys_II_Targaryen', 'Lyanna_Stark', 'Brandon_Stark_(son_of_Rickard)',
]

probe = []
for slug in PROBE_SLUGS:
    probe.append(extract_book_signals(slug))

probe_df = pd.DataFrame(probe)
probe_df['n_books']   = probe_df['books'].str.len()
probe_df['n_re']      = probe_df['recent_events_books'].str.len()
probe_df['n_history'] = probe_df['history_books'].str.len()
probe_df['n_h2']      = probe_df['h2_books'].str.len()
probe_df[['ID', 'n_books', 'n_re', 'n_history', 'n_h2', 'books']]

,ID,n_books,n_re,n_history,n_h2,books
0,Eddard_Stark,4,4,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
1,Catelyn_Stark,5,5,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
2,Tyrion_Lannister,6,6,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
3,Jon_Snow,5,5,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
4,Daenerys_Targaryen,5,5,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
5,Sansa_Stark,6,6,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
6,Arya_Stark,6,6,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
7,Bran_Stark,5,5,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
8,Cersei_Lannister,6,6,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."
9,Jaime_Lannister,5,5,0,0,"[A Game of Thrones, A Clash of Kings, A Storm ..."


**Sanity check:** look at the row for each character and confirm the book list matches your expectation.

Specific expectations:
- `Eddard_Stark` → AGOT (dies there)
- `Jon_Snow` → AGOT, ACOK, ASOS, ADWD (no AFFC)
- `Brienne_of_Tarth` → ACOK, ASOS, **AFFC**
- `Daenerys_Targaryen` → AGOT, ACOK, ASOS, ADWD (no AFFC)
- `Aegon_I_Targaryen` → Fire & Blood
- `Duncan_the_Tall` → The Hedge Knight, The Sworn Sword, The Mystery Knight
- `Rhaegar_Targaryen` → may have **zero** book entries — the wiki may or may not give him `History` subsections by book

If sanity-check holds, proceed to section 4.

## 4. Full scrape (3,690 characters)

**Run only after the probe looks correct.** Set `RUN_FULL = True` and execute. With 8 workers the scrape took ~10 min last time we ran the section-headings survey.

In [31]:
RUN_FULL = True  # flip to True to scrape all characters

if RUN_FULL:
    characters = pd.read_csv('../csvs/characters.csv')
    slugs = characters['ID'].tolist()
    print(f'Scraping {len(slugs)} characters...')
    results = []
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, r in enumerate(ex.map(extract_book_signals, slugs), 1):
            results.append(r)
            if i % 250 == 0:
                print(f'  {i}/{len(slugs)}')
    books_df = pd.DataFrame(results)
    print(f'Done. {len(books_df)} rows.')
else:
    print('Skipping full scrape. Set RUN_FULL = True to enable.')
    books_df = probe_df  # fall back to probe for downstream cells

Scraping 3690 characters...
  250/3690
  500/3690
  750/3690
  1000/3690
  1250/3690
  1500/3690
  1750/3690
  2000/3690
  2250/3690
  2500/3690
  2750/3690
  3000/3690
  3250/3690
  3500/3690
Done. 3690 rows.


## 5. Coverage diagnostics

Three questions:

1. **Coverage** — what fraction of characters have at least one canonical book?
2. **Per-book counts** — does the per-character count agree with the global section-heading survey (`scrape_section_headings.ipynb`)?
3. **Series purity** — how many characters appear in books from more than one series (e.g. *Fire & Blood* + ASOIAF)? If many do, then "primary series" is ambiguous and book-set is the right output, not a single label.


In [32]:
# 1. Coverage
total = len(books_df)
with_any = (books_df['books'].str.len() > 0).sum()
print(f'Characters with ≥ 1 canonical book section: {with_any}/{total} ({with_any/total:.1%})')

# 2. Per-book counts
book_counts = Counter()
for bs in books_df['books']:
    for b in bs:
        book_counts[b] += 1
print('\nPer-book character counts:')
for b, (series, order) in sorted(BOOK_CANON.items(), key=lambda kv: kv[1][1]):
    print(f'  [{series:13s}] {b:35s} {book_counts.get(b, 0):>5d}')

# 3. Series purity
def series_of(book_list):
    return {BOOK_CANON[b][0] for b in book_list}

books_df['series_set'] = books_df['books'].apply(series_of)
books_df['n_series']   = books_df['series_set'].str.len()
print('\nNumber of series each character spans:')
print(books_df['n_series'].value_counts().sort_index().to_string())

Characters with ≥ 1 canonical book section: 1883/3690 (51.0%)

Per-book character counts:
  [Fire & Blood ] Fire & Blood                            0
  [Fire & Blood ] The Princess and the Queen              0
  [Fire & Blood ] The Rogue Prince                        0
  [Fire & Blood ] The Sons of the Dragon                  1
  [Dunk & Egg   ] The Hedge Knight                        0
  [Dunk & Egg   ] The Sworn Sword                         3
  [Dunk & Egg   ] The Mystery Knight                      0
  [Dunk & Egg   ] The She-Wolves of Winterfell            0
  [ASOIAF       ] A Game of Thrones                     382
  [ASOIAF       ] A Clash of Kings                      595
  [ASOIAF       ] A Storm of Swords                     793
  [ASOIAF       ] A Feast for Crows                     830
  [ASOIAF       ] A Dance with Dragons                  813
  [ASOIAF       ] The Winds of Winter                   186

Number of series each character spans:
n_series
0    1807
1    1883


In [33]:
# Cross-series characters: who appears in books from more than one publication series?
cross = books_df[books_df['n_series'] > 1][['ID', 'books', 'series_set']]
print(f'{len(cross)} characters with cross-series book entries (first 30 shown):')
cross.head(30)

0 characters with cross-series book entries (first 30 shown):


,ID,books,series_set


## 6. Proposed labels

Based on the diagnostics above, pick one of the following labeling strategies:

- **`primary_series`** — single label per character (`Fire & Blood`, `Dunk & Egg`, `ASOIAF`, or `Unknown`). For cross-series characters, prefer the series with the most book entries; on a tie, prefer the later series (matches in-universe timeline ordering). Replaces `book_bucket` in `eras.csv`.
- **`first_book`** — the earliest book (by publication order) the character appears in. Per-book granularity for ASOIAF main, cleaner than date-based for cross-series. Use as a new column alongside `book_bucket`.
- **`book_set`** — keep the full list. Pure data, no rule needed. Use when you want set-based analysis (e.g. "characters in both ASOS and ADWD").

For network visualization in `04_timeline_plots.ipynb`, `primary_series` is the right replacement for the current `book_bucket` — it's still a single-label coloring but it's evidence-based (from explicit wiki annotations) rather than date-rule-based.

In [34]:
SERIES_ORDER = ['Fire & Blood', 'Dunk & Egg', 'ASOIAF']

def primary_series(book_list):
    if not book_list:
        return 'Unknown'
    counts = Counter(BOOK_CANON[b][0] for b in book_list)
    max_n = max(counts.values())
    tied = [s for s, n in counts.items() if n == max_n]
    # Tie-break: prefer the later series in canonical timeline ordering
    return sorted(tied, key=lambda s: SERIES_ORDER.index(s))[-1]

def first_book(book_list):
    if not book_list:
        return 'Unknown'
    return min(book_list, key=lambda b: BOOK_CANON[b][1])

books_df['primary_series'] = books_df['books'].apply(primary_series)
books_df['first_book']     = books_df['books'].apply(first_book)

print('primary_series distribution:')
print(books_df['primary_series'].value_counts().to_string())
print('\nfirst_book distribution (top 10):')
print(books_df['first_book'].value_counts().head(10).to_string())

primary_series distribution:
primary_series
ASOIAF          1879
Unknown         1807
Dunk & Egg         3
Fire & Blood       1

first_book distribution (top 10):
first_book
Unknown                   1807
A Feast for Crows          387
A Game of Thrones          382
A Storm of Swords          376
A Dance with Dragons       346
A Clash of Kings           345
The Winds of Winter         43
The Sworn Sword              3
The Sons of the Dragon       1


## 7. Compare against date-based `book_bucket` from `eras.csv`

If the two labels agree on 80%+ of dated characters, the book-section signal is at least as good as the date-based rule, and is far better for the Unknown bucket (where dates are missing but book sections often aren't).

In [35]:
import os
print(os.listdir())
if RUN_FULL and os.path.exists('eras.csv'):
    eras = pd.read_csv('eras.csv')
    print(eras)
    if 'book_bucket' in eras.columns:
        merged = books_df.merge(eras[['ID', 'book_bucket']], on='ID', how='inner')
        # Normalize names so they compare cleanly
        BUCKET_TO_SERIES = {
            'Fire & Blood': 'Fire & Blood',
            'Tales of Dunk & Egg': 'Dunk & Egg',
            'A Song of Ice and Fire': 'ASOIAF',
            'Unknown': 'Unknown',
        }
        merged['date_series'] = merged['book_bucket'].map(BUCKET_TO_SERIES)
        both_known = merged[(merged['primary_series'] != 'Unknown') & (merged['date_series'] != 'Unknown')]
        agree = (both_known['primary_series'] == both_known['date_series']).sum()
        print(f'Agreement on {len(both_known)} doubly-labeled characters: {agree} ({agree/len(both_known):.1%})')
        print('\nConfusion matrix (rows=book-section, cols=date-based):')
        print(pd.crosstab(both_known['primary_series'], both_known['date_series']))
    else:
        print('eras.csv has no book_bucket column — rerun 03_execute.ipynb first.')
else:
    print('Skipping comparison (RUN_FULL=False or eras.csv missing).')

['02_plan.ipynb', 'eras.csv', '01_research.ipynb', '05_book_segmentation_research.ipynb', '03_execute.ipynb', '04_timeline_plots.ipynb', 'dates.csv']
                       ID   year   source      era
0           A_certain_man    NaN  unknown  Unknown
1        Abelar_Hightower    NaN  unknown  Unknown
2                  Abelon    NaN  unknown  Unknown
3     Addam_of_Duskendale    NaN  unknown  Unknown
4              Addam_Frey    NaN  unknown  Unknown
...                   ...    ...      ...      ...
3685      Zharaq_zo_Loraq    NaN  unknown  Unknown
3686                 Zhea    NaN  unknown  Unknown
3687       Zhoe_Blanetree    NaN  unknown  Unknown
3688             Zia_Frey  315.0  born+30  Unknown
3689                Zollo    NaN  unknown  Unknown

[3690 rows x 4 columns]
eras.csv has no book_bucket column — rerun 03_execute.ipynb first.


## Next step

If the probe + full scrape confirm good coverage and reasonable agreement with the date-based bucket:

1. Save `books_df` as `book_signals.csv` (columns: `ID`, `books`, `primary_series`, `first_book`).
2. Update `03_execute.ipynb` to merge `book_signals.csv` into `eras.csv` and add `primary_series` + `first_book` columns alongside `era` and `book_bucket`.
3. Update `04_timeline_plots.ipynb` plot 1 to colour by `primary_series` (evidence-based) instead of `book_bucket` (date-rule-based). For Plot 2 (ASOIAF main only) we can sub-segment by `first_book` to show *which* book a character debuts in.